In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("STEP 1: Job parameters received")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")

In [0]:
source_table = f"databricks_caresync.bronze.{table_name}"
filter_condition = f"landing_timestamp='{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading source batch from bronze")
print("=" * 80)
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print("  Equivalent SQL query:")
print(f"    SELECT * FROM {source_table}\n    WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"  Records read from bronze : {bronze_count}")
print(f"  Source columns ({len(bronze_df.columns)}): {bronze_df.columns}")
if bronze_count == 0:
    print("  WARNING: No records found for this landing_timestamp. Check the parameter value.")

In [0]:
from pyspark.sql import functions as F

print("=" * 80)
print("STEP 3: Adding audit columns")
print("=" * 80)
print("  insert_timestamp = current_timestamp()")
print("  update_timestamp = current_timestamp()")

updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())

print(f"  Output columns ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f"databricks_caresync.silver.{table_name}"
load_type_value = load_type.strip().upper()
column_list = ", ".join(f"`{c}`" for c in updated_df.columns)

print("=" * 80)
print("STEP 4: Loading data into silver")
print("=" * 80)
print(f"  Target table : {target_table}")
print(f"  Load type    : {load_type_value or '<empty>'}")
print(f"  Target table exists before load : {spark.catalog.tableExists(target_table)}")

if load_type_value == "FULL":
    # Full load: replace the target table contents
    print("  Action : FULL load - overwriting target table")
    print("  Equivalent SQL query:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS\n    SELECT {column_list}\n    FROM <source_batch>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed.")

elif load_type_value == "APPEND":
    # Append: add the new batch to the target table
    print("  Action : APPEND - inserting batch into target table")
    print("  Equivalent SQL query:")
    print(f"    INSERT INTO {target_table} ({column_list})\n    SELECT {column_list}\n    FROM <source_batch>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND completed.")

elif load_type_value == "MERGE":
    # Merge: update existing records and insert new ones based on merge keys
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type is MERGE")
    missing_keys = [k for k in keys if k not in updated_df.columns]
    if missing_keys:
        raise ValueError(f"Merge keys not found in source data: {missing_keys}")
    print(f"  Merge keys : {keys}")

    if not spark.catalog.tableExists(target_table):
        # First run: create the target table
        print("  Target table does not exist - creating it from the source batch (initial load).")
        print("  Equivalent SQL query:")
        print(f"    CREATE TABLE {target_table} AS\n    SELECT {column_list}\n    FROM <source_batch>")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial load completed.")
    else:
        # Deduplicate the source on merge keys so each target row matches at most one source row
        source_df = updated_df.dropDuplicates(keys)
        source_count = updated_df.count()
        dedup_count = source_df.count()
        print(f"  Source records : {source_count} | after dedup on keys : {dedup_count} | duplicates removed : {source_count - dedup_count}")
        merge_condition = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
        # On update keep the original insert_timestamp; refresh all other columns incl. update_timestamp
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_values = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        update_clause = ",\n        ".join(f"t.{k} = {v}" for k, v in update_set.items())
        insert_cols = ", ".join(insert_values.keys())
        insert_vals = ", ".join(insert_values.values())
        print(f"  Merge condition : {merge_condition}")
        print("  Equivalent SQL query:")
        print(
            f"    MERGE INTO {target_table} AS t\n"
            f"    USING <source_batch> AS s\n"
            f"    ON {merge_condition}\n"
            f"    WHEN MATCHED THEN UPDATE SET\n        {update_clause}\n"
            f"    WHEN NOT MATCHED THEN INSERT ({insert_cols})\n"
            f"    VALUES ({insert_vals})"
        )

        (
            DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_values)
            .execute()
        )

        # Report merge metrics from the Delta transaction log
        metrics = (
            DeltaTable.forName(spark, target_table).history(1)
            .select("operation", "operationMetrics").collect()[0]
        )
        op_metrics = metrics["operationMetrics"] or {}
        print(f"  MERGE completed. Operation: {metrics['operation']}")
        print(f"    Rows inserted : {op_metrics.get('numTargetRowsInserted', 'n/a')}")
        print(f"    Rows updated  : {op_metrics.get('numTargetRowsUpdated', 'n/a')}")
        print(f"    Rows deleted  : {op_metrics.get('numTargetRowsDeleted', 'n/a')}")

else:
    print(f"  ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

print(f"  Target record count after load : {spark.table(target_table).count()}")
print("=" * 80)
print("Load finished successfully.")
print("=" * 80)

In [0]:
%sql
select * from databricks_caresync.silver.hospitals

In [0]:
print("=" * 80)
print("[STEP 5] Computing processed record count")
print("=" * 80)

record_count = updated_df.count()

print(f"[STEP 5 COMPLETE] Processed record count: {record_count}")

dbutils.notebook.exit(str(record_count))